In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import os
import numpy as np
import json
from collections import Counter
from numpy import linalg as la
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import OPTICS
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score
import importlib
import model as model_module
importlib.reload(model_module)
LogKG = model_module.LogKG

# OS_preprocessed dataset config
# Default to the full B507 preprocessed dataset; allow env override when needed.
DEFAULT_OS_PREPROCESSED_DIR = os.path.join("..", "data", "OS_preprocessed_B507")
OS_PREPROCESSED_DIR = os.environ.get("LOGKG_OS_PREPROCESSED_DIR", DEFAULT_OS_PREPROCESSED_DIR)
OS_PREPROCESSED_DIR = os.path.normpath(OS_PREPROCESSED_DIR)
OS_CASE_DIR = os.path.join(OS_PREPROCESSED_DIR, "cases")
OS_CONFIG_MAJOR_PATH = os.path.join(OS_PREPROCESSED_DIR, "config_major.json")
OS_CONFIG_MINOR_PATH = os.path.join(OS_PREPROCESSED_DIR, "config_minor.json")
OS_CONFIG_PAIR_PATH = os.path.join(OS_PREPROCESSED_DIR, "config_pair.json")
print("OS_PREPROCESSED_DIR:", OS_PREPROCESSED_DIR)

from sklearn.feature_extraction.text import HashingVectorizer




OS_PREPROCESSED_DIR: ..\data\OS_preprocessed_B507


In [2]:
# Core experiment config（核心实验配置）

EMBEDDING_SIZE = 16
# 嵌入向量维度（embedding dimension），每个日志模板会被表示成16维向量

template_embedding = None
# 模板嵌入变量，占位用（用于存储 EventTemplate 的 embedding）

# Label granularity for OS_preprocessed training（标签粒度设置）
# - 'major': 使用 major_problem_type（主要故障类型，推荐先用这个）
# - 'minor': 使用 minor_problem_type（细粒度问题类型，类别长尾更严重）
# - 'pair' : major||minor（主+次问题类型组合标签）
LABEL_LEVEL = "minor"

# Template embedding mode（日志模板向量化方式）
# - 'random'   : 随机向量（当前稳定 baseline）
# - 'hash_text': 将 EventTemplate 文本 hash 成向量
TEMPLATE_EMBEDDING_MODE = "random"

# Eval mode（评估模式）
# - 'supervised': 直接使用真实标签训练 Random Forest（推荐）
# - 'logkg'     : 原始 LogKG 流程（先聚类生成伪标签，再训练 RF）
EVAL_MODE = "supervised"

# Rare class handling for StratifiedKFold（处理少样本类别）
MIN_SAMPLES_PER_CLASS = 2
# 每个类别最少需要的样本数（否则无法做分层交叉验证）

RARE_CLASS_STRATEGY = "drop"
# 稀有类别处理策略
# "drop" 表示删除样本数量少于 MIN_SAMPLES_PER_CLASS 的类别


# Intermediate trace config
# Keep False by default for the full B507 dataset; enable only when debugging one run.
SHOW_INTERMEDIATE_TRACE = False
TRACE_FOLD_INDEX = 1
TRACE_CASES_PER_SPLIT = 2
TRACE_TOP_TEMPLATES = 10

# Case loading progress
LOAD_PROGRESS_EVERY = 500


In [3]:
# ===============================
# 根据 case 名称生成 ground truth label（真实标签）
# ===============================
def get_case_truth_label(case_name_list, input_config):

    # 创建标签数组，长度 = case 数量，初始值为 0
    truth_label_list = np.zeros((len(case_name_list)), dtype=int)

    # 建立 case_name → index 的映射
    # 这样可以 O(1) 查找，避免 list.index 的 O(n²) 时间复杂度
    case_index_map = {name: idx for idx, name in enumerate(case_name_list)}

    # 存储 “故障类别 → 数字标签” 的映射
    label_config = {}

    # 遍历 input_config（配置文件中的故障类别）
    for index, fault_class in enumerate(input_config.keys()):

        # 为每个故障类别分配一个整数标签
        label_config[fault_class] = index

        # 遍历该故障类别下的所有 case
        for case_name in input_config[fault_class]:

            # 如果配置里的 case 不在 case_name_list 中，报错
            if case_name not in case_index_map:
                raise ValueError(f"case in config but missing csv file: {case_name}")

            # 将该 case 对应位置的标签设置为当前故障类别 index
            truth_label_list[case_index_map[case_name]] = index

    # 返回每个 case 对应的真实标签数组
    return truth_label_list

In [4]:
# ===============================
# 聚类中一个簇允许的最少样本数量
# ===============================
# 如果一个簇中的样本数 < 3，
# OPTICS 可能会把这些样本视为噪声点（label = -1）
min_samples = 3


# ===============================
# 构建聚类模型（Cluster Model）
# ===============================
def build_cluster_model():
    return OPTICS(
        min_samples=min_samples,   # 每个簇至少要有 3 个样本
        metric="cosine",           # 使用余弦距离衡量样本相似度
        xi=0.05,                   # 控制簇边界敏感度，越小划分越细
        algorithm="brute"          # 暴力计算距离，适合小规模数据
    )


# ===============================
# IDF 阈值（过滤低信息模板）
# ===============================
# 在 LogKG 中：
# IDF = log10(case总数 / 包含该模板的case数)
#
# 如果某个模板的 IDF < 0.4，
# 说明它在很多 case 中都出现，区分能力较弱，
# 会被视为低信息模板并过滤掉（权重设为 0）

IDF_threshold = 0.4


# ===============================
# 分类器（Classifier）
# ===============================
# LogKG 会先通过聚类生成伪标签（pseudo-label），
# 再使用监督分类器进行训练

clf = RandomForestClassifier


# 分类器参数
clf_kwargs = {
    "random_state": 42,                 # 固定随机种子，保证结果可复现
    "n_estimators": 600,                # 随机森林中树的数量
    "class_weight": "balanced_subsample", # 对类别不平衡进行加权
    "n_jobs": 1,                        # 使用 1 个 CPU 核心训练
}

In [5]:
#对训练样本做聚类，然后从每个簇中
#选一个“最像这个簇中心的真实样本”作为代表

# ==================================================
# 1) EDM: Euclidean Distance Matrix
# ==================================================
def compute_squared_EDM_method(X):
    n, _ = X.shape
    D = np.zeros([n, n])

    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = la.norm(X[i, :] - X[j, :])
            D[j, i] = D[i, j]
    return D


# ==================================================
# 2) Cluster center index (medoid-like)
# ==================================================
def get_centroid_index(cluster_embedding):
    distance_array = np.sum(compute_squared_EDM_method(cluster_embedding), axis=1)
    return np.argmin(distance_array)


# ==================================================
# 3) Train clustering + choose one representative per cluster
# ==================================================
def get_logkg_result(train_set, train_index, verbose=False):
    cluster_model = build_cluster_model()
    cluster_ = cluster_model.fit_predict(train_set)
    class_num = np.max(cluster_) + 1

    if verbose:
        print(cluster_)
        print(Counter(cluster_))
        print('class_num: {}'.format(class_num))

    centroids = [
        train_index[
            np.where(cluster_ == i)[0][
                get_centroid_index(train_set[np.where(cluster_ == i)[0]])
            ]
        ]
        for i in range(class_num)
    ]
    return centroids, cluster_


In [6]:
# ==================================================
# 4) Single fold: LogKG -> pseudo labels -> classifier
#先对训练日志 case 聚类，再用每个簇的代表样本真实标签生成伪标签，
# 最后训练分类器并在测试集上评估性能。
# ==================================================
def LogKG_exp_run(case_name_list, case_truth_label,
                  train_index, test_index,
                  logkg_config, verbose=False):
    logkg = logkg_config

    if len(train_index) == 0 or len(test_index) == 0:
        raise ValueError('train_index/test_index is empty. Please set a non-empty split before running.')

    logkg.get_train_embedding()
    logkg.get_test_embedding()
    train_embedding = logkg.train_embedding_dict
    test_embedding = logkg.test_embedding_dict

    train_set = np.array([train_embedding[case_name_list[index]] for index in train_index])
    test_set = np.array([test_embedding[case_name_list[index]] for index in test_index])

    cluster_centroids, cluster_result = get_logkg_result(
        train_set=train_set,
        train_index=train_index,
        verbose=verbose,
    )

    classify_index = np.zeros(len(cluster_result), dtype=int) - 1

    for i in range(np.max(cluster_result) + 1):
        class_label = case_truth_label[cluster_centroids[i]]
        classify_index[np.where(cluster_result == i)[0]] = class_label

    classify_train_mask = classify_index != -1
    if not np.any(classify_train_mask):
        raise ValueError('No labeled samples after clustering.')

    classify_train_label = classify_index[classify_train_mask]
    classify_train_set = train_set[classify_train_mask]

    classifier = clf(**clf_kwargs)
    classifier.fit(classify_train_set, classify_train_label)

    y_true = case_truth_label[test_index].astype(int)
    y_pred = classifier.predict(test_set).astype(int)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    return y_true, y_pred, acc, macro_f1


In [7]:
# ==================================================
# 5) 分层 K 折交叉验证评估（LogKG 模式）
# ==================================================

def _calc_eval_metrics(y_true, y_pred):
    # 计算常用分类评估指标
    return {
        'acc': float(accuracy_score(y_true, y_pred)),  # 准确率 Accuracy
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),  # 宏平均 F1
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),  # 加权 F1
        'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),  # 宏平均 Recall
        'weighted_recall': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),  # 加权 Recall
    }


def run_stratified_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              verbose_fold=False,
                              min_samples_per_class=2,
                              rare_class_strategy='drop'):
    # 检查 case 名称列表和标签数组长度是否一致
    if len(case_name_list) != len(case_truth_label):
        raise ValueError('case_name_list and case_truth_label length mismatch')

    # 统计每个类别的样本数量
    class_counts = Counter(case_truth_label.tolist())

    # 找出样本数小于阈值的稀有类别
    rare_labels = [label for label, count in class_counts.items() if count < min_samples_per_class]

    # 如果存在稀有类别
    if rare_labels:
        if rare_class_strategy == 'drop':
            # 删除稀有类别对应的样本
            keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
            dropped_count = len(case_name_list) - len(keep_indices)

            # 如果全删光了，直接报错
            if len(keep_indices) == 0:
                raise ValueError('All samples are dropped by rare class filtering.')

            # 保留非稀有类别样本
            case_name_list = [case_name_list[idx] for idx in keep_indices]
            case_truth_label = case_truth_label[keep_indices]
            case_log_df = {name: case_log_df[name] for name in case_name_list}

            # 更新类别分布
            class_counts = Counter(case_truth_label.tolist())

            # 打印删除信息
            print(
                f'[Warn] Dropped {dropped_count} samples from rare classes '
                f'(labels={rare_labels}, min_samples_per_class={min_samples_per_class}).'
            )
            print(f'[Info] Remaining class distribution: {class_counts}')
        else:
            # 如果不允许 drop，则直接报错
            raise ValueError(
                f'Not enough samples per class for StratifiedKFold: '
                f'rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}'
            )

    # 分层 K 折要求：每个类别至少要有 >= n_splits 个样本
    # 所以实际 folds 数量不能超过最小类别样本数
    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)

    # 至少要能分成 2 折
    if effective_splits < 2:
        raise ValueError(
            f'Not enough samples per class for StratifiedKFold after filtering: '
            f'min_class_count={min_class_count}.'
        )

    # 创建分层 K 折对象
    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    # 每折指标列表
    acc_list, macro_f1_list = [], []
    weighted_f1_list, macro_recall_list, weighted_recall_list = [], [], []

    # 保存所有折的真实标签和预测标签，便于最后总体统计
    y_true_all, y_pred_all = [], []

    # 所有可能类别标签（用于 confusion matrix 保持顺序一致）
    labels = np.unique(case_truth_label)

    # 保存每一折的混淆矩阵
    fold_confusion_matrices = []

    # 开始分层 K 折
    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1,
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        # 构造当前折的训练/测试日志数据
        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        # 构造 LogKG 模型
        # 有的 LogKG 版本支持 embedding_size，有的版本不支持，所以做兼容处理
        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        # 执行单折实验：聚类 -> 伪标签 -> 分类器 -> 测试预测
        y_true, y_pred, _, _ = LogKG_exp_run(
            case_name_list=case_name_list,
            case_truth_label=case_truth_label,
            train_index=train_index,
            test_index=test_index,
            logkg_config=model,
            verbose=verbose_fold,
        )

        # 计算该折指标
        m = _calc_eval_metrics(y_true, y_pred)
        acc_list.append(m['acc'])
        macro_f1_list.append(m['macro_f1'])
        weighted_f1_list.append(m['weighted_f1'])
        macro_recall_list.append(m['macro_recall'])
        weighted_recall_list.append(m['weighted_recall'])

        # 保存该折预测结果
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        # 计算该折混淆矩阵
        fold_cm = confusion_matrix(y_true, y_pred, labels=labels)
        fold_confusion_matrices.append(fold_cm)

        # 打印当前 fold 结果
        print(
            f"fold {fold_idx}/{effective_splits}: "
            f"acc={m['acc']:.4f}, macro_f1={m['macro_f1']:.4f}, weighted_f1={m['weighted_f1']:.4f}, "
            f"macro_recall={m['macro_recall']:.4f}, weighted_recall={m['weighted_recall']:.4f}, test_size={len(test_index)}"
        )
        print(f'fold {fold_idx} confusion_matrix labels: {labels.tolist()}')
        print(fold_cm)

    # 拼接所有 fold 的预测结果，做整体统计
    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    # 计算各指标的均值和标准差
    acc_mean, acc_std = float(np.mean(acc_list)), float(np.std(acc_list))
    macro_f1_mean, macro_f1_std = float(np.mean(macro_f1_list)), float(np.std(macro_f1_list))
    weighted_f1_mean, weighted_f1_std = float(np.mean(weighted_f1_list)), float(np.std(weighted_f1_list))
    macro_recall_mean, macro_recall_std = float(np.mean(macro_recall_list)), float(np.std(macro_recall_list))
    weighted_recall_mean, weighted_recall_std = float(np.mean(weighted_recall_list)), float(np.std(weighted_recall_list))

    # 所有 fold 合并后的总体混淆矩阵
    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    # 打印最终总结
    print('=' * 30, 'StratifiedKFold Summary', '=' * 30)
    print(f'accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}')
    print(f'macro_f1: mean={macro_f1_mean:.4f}, std={macro_f1_std:.4f}')
    print(f'weighted_f1: mean={weighted_f1_mean:.4f}, std={weighted_f1_std:.4f}')
    print(f'macro_recall: mean={macro_recall_mean:.4f}, std={macro_recall_std:.4f}')
    print(f'weighted_recall: mean={weighted_recall_mean:.4f}, std={weighted_recall_std:.4f}')
    print('confusion_matrix labels:', labels.tolist())
    print(cm)

    # 返回完整评估结果字典
    return {
        'acc_list': acc_list,  # 每折 accuracy
        'macro_f1_list': macro_f1_list,  # 每折 macro F1
        'f1_list': macro_f1_list,  # 兼容旧字段名
        'weighted_f1_list': weighted_f1_list,  # 每折 weighted F1
        'macro_recall_list': macro_recall_list,  # 每折 macro recall
        'weighted_recall_list': weighted_recall_list,  # 每折 weighted recall

        'acc_mean': acc_mean,  # accuracy 均值
        'acc_std': acc_std,  # accuracy 标准差

        'macro_f1_mean': macro_f1_mean,  # macro F1 均值
        'f1_mean': macro_f1_mean,  # 兼容旧字段名
        'macro_f1_std': macro_f1_std,  # macro F1 标准差
        'f1_std': macro_f1_std,  # 兼容旧字段名

        'weighted_f1_mean': weighted_f1_mean,  # weighted F1 均值
        'weighted_f1_std': weighted_f1_std,  # weighted F1 标准差

        'macro_recall_mean': macro_recall_mean,  # macro recall 均值
        'macro_recall_std': macro_recall_std,  # macro recall 标准差

        'weighted_recall_mean': weighted_recall_mean,  # weighted recall 均值
        'weighted_recall_std': weighted_recall_std,  # weighted recall 标准差

        'labels': labels,  # 类别标签列表
        'fold_confusion_matrices': fold_confusion_matrices,  # 每折混淆矩阵
        'confusion_matrix': cm,  # 总体混淆矩阵
        'effective_splits': effective_splits,  # 实际使用的折数
        'final_class_dist': dict(class_counts),  # 最终类别分布
    }

In [8]:
# ==================================================
# 6) StratifiedKFold evaluation (Supervised mode)
# ==================================================
def run_supervised_kfold_eval(case_name_list, case_truth_label,
                              case_log_df, template_embedding,
                              n_splits=5, random_state=42,
                              min_samples_per_class=2,
                              rare_class_strategy='drop'):
    if len(case_name_list) != len(case_truth_label):
        raise ValueError('case_name_list and case_truth_label length mismatch')

    class_counts = Counter(case_truth_label.tolist())
    rare_labels = [label for label, count in class_counts.items() if count < min_samples_per_class]

    if rare_labels:
        if rare_class_strategy == 'drop':
            keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
            dropped_count = len(case_name_list) - len(keep_indices)
            if len(keep_indices) == 0:
                raise ValueError('All samples are dropped by rare class filtering.')
            case_name_list = [case_name_list[idx] for idx in keep_indices]
            case_truth_label = case_truth_label[keep_indices]
            case_log_df = {name: case_log_df[name] for name in case_name_list}
            class_counts = Counter(case_truth_label.tolist())
            print(
                f'[Warn] Dropped {dropped_count} samples from rare classes '
                f'(labels={rare_labels}, min_samples_per_class={min_samples_per_class}).'
            )
            print(f'[Info] Remaining class distribution: {class_counts}')
        else:
            raise ValueError(
                f'Not enough samples per class for StratifiedKFold: '
                f'rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}'
            )

    min_class_count = min(class_counts.values())
    effective_splits = min(n_splits, min_class_count)
    if effective_splits < 2:
        raise ValueError(
            f'Not enough samples per class for StratifiedKFold after filtering: '
            f'min_class_count={min_class_count}.'
        )

    skf = StratifiedKFold(n_splits=effective_splits, shuffle=True, random_state=random_state)

    acc_list, macro_f1_list = [], []
    weighted_f1_list, macro_recall_list, weighted_recall_list = [], [], []
    y_true_all, y_pred_all = [], []
    labels = np.unique(case_truth_label)
    fold_confusion_matrices = []

    for fold_idx, (train_index, test_index) in enumerate(
        skf.split(np.arange(len(case_name_list)), case_truth_label),
        start=1,
    ):
        train_index = train_index.tolist()
        test_index = test_index.tolist()

        train_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in train_index}
        test_df = {case_name_list[idx]: case_log_df[case_name_list[idx]] for idx in test_index}

        try:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
        except TypeError:
            model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

        model.get_train_embedding()
        model.get_test_embedding()

        train_set = np.array([model.train_embedding_dict[case_name_list[idx]] for idx in train_index])
        test_set = np.array([model.test_embedding_dict[case_name_list[idx]] for idx in test_index])

        y_train = case_truth_label[train_index].astype(int)
        y_true = case_truth_label[test_index].astype(int)

        classifier = clf(**clf_kwargs)
        classifier.fit(train_set, y_train)
        y_pred = classifier.predict(test_set).astype(int)

        m = _calc_eval_metrics(y_true, y_pred)
        acc_list.append(m['acc'])
        macro_f1_list.append(m['macro_f1'])
        weighted_f1_list.append(m['weighted_f1'])
        macro_recall_list.append(m['macro_recall'])
        weighted_recall_list.append(m['weighted_recall'])
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)

        fold_cm = confusion_matrix(y_true, y_pred, labels=labels)
        fold_confusion_matrices.append(fold_cm)

        print(
            f"fold {fold_idx}/{effective_splits}: "
            f"acc={m['acc']:.4f}, macro_f1={m['macro_f1']:.4f}, weighted_f1={m['weighted_f1']:.4f}, "
            f"macro_recall={m['macro_recall']:.4f}, weighted_recall={m['weighted_recall']:.4f}, test_size={len(test_index)}"
        )
        print(f'fold {fold_idx} confusion_matrix labels: {labels.tolist()}')
        print(fold_cm)

    y_true_all = np.concatenate(y_true_all).astype(int)
    y_pred_all = np.concatenate(y_pred_all).astype(int)

    acc_mean, acc_std = float(np.mean(acc_list)), float(np.std(acc_list))
    macro_f1_mean, macro_f1_std = float(np.mean(macro_f1_list)), float(np.std(macro_f1_list))
    weighted_f1_mean, weighted_f1_std = float(np.mean(weighted_f1_list)), float(np.std(weighted_f1_list))
    macro_recall_mean, macro_recall_std = float(np.mean(macro_recall_list)), float(np.std(macro_recall_list))
    weighted_recall_mean, weighted_recall_std = float(np.mean(weighted_recall_list)), float(np.std(weighted_recall_list))

    cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

    print('=' * 30, 'Supervised StratifiedKFold Summary', '=' * 30)
    print(f'accuracy: mean={acc_mean:.4f}, std={acc_std:.4f}')
    print(f'macro_f1: mean={macro_f1_mean:.4f}, std={macro_f1_std:.4f}')
    print(f'weighted_f1: mean={weighted_f1_mean:.4f}, std={weighted_f1_std:.4f}')
    print(f'macro_recall: mean={macro_recall_mean:.4f}, std={macro_recall_std:.4f}')
    print(f'weighted_recall: mean={weighted_recall_mean:.4f}, std={weighted_recall_std:.4f}')
    print('confusion_matrix labels:', labels.tolist())
    print(cm)

    return {
        'acc_list': acc_list,
        'macro_f1_list': macro_f1_list,
        'f1_list': macro_f1_list,
        'weighted_f1_list': weighted_f1_list,
        'macro_recall_list': macro_recall_list,
        'weighted_recall_list': weighted_recall_list,
        'acc_mean': acc_mean,
        'acc_std': acc_std,
        'macro_f1_mean': macro_f1_mean,
        'f1_mean': macro_f1_mean,
        'macro_f1_std': macro_f1_std,
        'f1_std': macro_f1_std,
        'weighted_f1_mean': weighted_f1_mean,
        'weighted_f1_std': weighted_f1_std,
        'macro_recall_mean': macro_recall_mean,
        'macro_recall_std': macro_recall_std,
        'weighted_recall_mean': weighted_recall_mean,
        'weighted_recall_std': weighted_recall_std,
        'labels': labels,
        'fold_confusion_matrices': fold_confusion_matrices,
        'confusion_matrix': cm,
        'effective_splits': effective_splits,
        'final_class_dist': dict(class_counts),
    }


In [9]:
# ================================
# 1️⃣ 标签层级配置（label level config）
# ================================
label_level_to_config = {
    "major": OS_CONFIG_MAJOR_PATH,   # 粗粒度标签（major level）
    "minor": OS_CONFIG_MINOR_PATH,   # 中粒度标签（minor level）
    "pair": OS_CONFIG_PAIR_PATH,     # 组合标签（pair level）
}

# 检查 LABEL_LEVEL 是否合法
if LABEL_LEVEL not in label_level_to_config:
    raise ValueError(f"Unsupported LABEL_LEVEL: {LABEL_LEVEL}. Choose from {list(label_level_to_config.keys())}.")

# 根据标签层级选择配置文件路径
config_path = label_level_to_config[LABEL_LEVEL]
case_dir = OS_CASE_DIR  # case 数据目录

# ================================
# 2️⃣ 路径检查（sanity check）
# ================================
if not os.path.exists(config_path):
    raise FileNotFoundError(f"config not found: {config_path}")
if not os.path.exists(case_dir):
    raise FileNotFoundError(f"case dir not found: {case_dir}")

# ================================
# 3️⃣ 获取 case 列表 & 标签（labels）
# ================================
# 获取所有 case 名（去掉 .csv 后缀）
case_name_list = sorted([
    os.path.splitext(name)[0]
    for name in os.listdir(case_dir)
    if name.endswith(".csv")
])

# 加载标签配置
config = json.load(open(config_path, "r", encoding="utf-8"))

# 获取每个 case 对应的真实标签（ground truth label）
case_truth_label = get_case_truth_label(case_name_list, config)

# ================================
# 4️⃣ 读取每个 case 的日志（log dataframe）
# ================================
case_log_df = {}

for idx, case_name in enumerate(case_name_list, start=1):
    df_case = pd.read_csv(
        os.path.join(case_dir, case_name + ".csv"),
        dtype={"EventId": "string"}  # 强制 EventId 为字符串
    )

    # 检查是否有 EventId 列（关键字段）
    if "EventId" not in df_case.columns:
        raise ValueError(f"EventId column missing in case file: {case_name}.csv")

    # 缺失值填充（NO_LOG）并转字符串
    df_case["EventId"] = df_case["EventId"].fillna("NO_LOG").astype(str)

    case_log_df[case_name] = df_case  # 存入 dict（case -> dataframe）
    if idx % LOAD_PROGRESS_EVERY == 0 or idx == len(case_name_list):
        print(f"loaded cases: {idx}/{len(case_name_list)}")

# ================================
# 5️⃣ 构建 template embedding（模板向量）
# ================================
def build_template_embedding(case_log_df, embedding_size=128, mode="hash_text", seed=42):
    """
    输入：所有 case 的日志
    输出：每个 EventId 对应一个 embedding 向量

    mode:
    - "random"    : 随机向量（baseline）
    - "hash_text" : 基于文本 hashing（推荐）
    """

    # ----------------------------
    # 🟢 模式1：随机 embedding（random baseline）
    # ----------------------------
    if mode == "random":
        rng = np.random.default_rng(seed)

        # 收集所有 template（EventId）
        all_templates = sorted({
            str(eid)
            for df in case_log_df.values()
            for eid in df["EventId"].dropna().values
            if str(eid) != ""
        })

        # 为每个 template 分配随机向量
        return {eid: rng.normal(0, 1, embedding_size) for eid in all_templates}

    # ----------------------------
    # 🟢 模式2：基于文本 hashing（核心）
    # ----------------------------
    eventid_to_template = {}

    for df in case_log_df.values():
        if "EventTemplate" in df.columns:
            # 如果有模板文本（推荐）
            for eid, tpl in zip(
                df["EventId"].astype(str).tolist(),
                df["EventTemplate"].astype(str).tolist()
            ):
                if eid and eid not in eventid_to_template:
                    eventid_to_template[eid] = tpl
        else:
            # fallback：没有模板，用 EventId 本身
            for eid in df["EventId"].astype(str).tolist():
                if eid and eid not in eventid_to_template:
                    eventid_to_template[eid] = eid

    # 所有 template id
    all_ids = sorted(eventid_to_template.keys())

    # 对应文本（template text）
    texts = [eventid_to_template[eid] for eid in all_ids]

    # ----------------------------
    # 🟢 HashingVectorizer（文本 -> 向量）
    # ----------------------------
    vectorizer = HashingVectorizer(
        n_features=embedding_size,   # 向量维度
        alternate_sign=False,        # 不使用正负号
        norm=None,                   # 不做归一化（后面手动）
        token_pattern=r"(?u)\b\w+\b" # token 切分规则
    )

    # 转为稠密矩阵
    mat = vectorizer.transform(texts).toarray().astype(float)

    # ----------------------------
    # 🟢 L2 归一化（normalize）
    # ----------------------------
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0  # 防止除0
    mat = mat / norms

    # 返回 dict：EventId -> embedding vector
    return {eid: mat[idx] for idx, eid in enumerate(all_ids)}

# ================================
# 6️⃣ 初始化 template embedding（只执行一次）
# ================================
if template_embedding is None:
    template_embedding = build_template_embedding(
        case_log_df=case_log_df,
        embedding_size=EMBEDDING_SIZE,
        mode=TEMPLATE_EMBEDDING_MODE,
        seed=42,
    )
    print("template_embedding initialized:", len(template_embedding), "mode=", TEMPLATE_EMBEDDING_MODE)

# ================================
# 7️⃣ 打印数据集信息（dataset info）
# ================================
print("DATASET:", os.path.basename(OS_PREPROCESSED_DIR))
print("LABEL_LEVEL:", LABEL_LEVEL)
print("config_path:", config_path)
print("case_dir:", case_dir)
print("case_num:", len(case_name_list))

# 打印类别分布（class distribution）
print("class_dist:", Counter(case_truth_label.tolist()))

loaded cases: 500/7121
loaded cases: 1000/7121
loaded cases: 1500/7121
loaded cases: 2000/7121
loaded cases: 2500/7121
loaded cases: 3000/7121
loaded cases: 3500/7121
loaded cases: 4000/7121
loaded cases: 4500/7121
loaded cases: 5000/7121
loaded cases: 5500/7121
loaded cases: 6000/7121
loaded cases: 6500/7121
loaded cases: 7000/7121
loaded cases: 7121/7121
template_embedding initialized: 331508 mode= random
DATASET: OS_preprocessed_B507
LABEL_LEVEL: minor
config_path: ..\data\OS_preprocessed_B507\config_minor.json
case_dir: ..\data\OS_preprocessed_B507\cases
case_num: 7121
class_dist: Counter({7: 1459, 1: 965, 4: 841, 6: 767, 2: 741, 8: 396, 25: 295, 3: 240, 9: 200, 5: 178, 0: 149, 16: 137, 18: 98, 17: 95, 10: 92, 20: 91, 12: 89, 19: 82, 24: 62, 13: 61, 11: 28, 21: 15, 15: 10, 27: 9, 14: 7, 22: 7, 23: 3, 26: 2, 28: 2})


In [10]:
print('-' * 30, f"Eval Mode: {EVAL_MODE}", '-' * 30)
if EVAL_MODE == "supervised":
    eval_result = run_supervised_kfold_eval(
        case_name_list=case_name_list,
        case_truth_label=case_truth_label,
        case_log_df=case_log_df,
        template_embedding=template_embedding,
        n_splits=5,
        random_state=42,
        min_samples_per_class=MIN_SAMPLES_PER_CLASS,
        rare_class_strategy=RARE_CLASS_STRATEGY,
    )
elif EVAL_MODE == "logkg":
    eval_result = run_stratified_kfold_eval(
        case_name_list=case_name_list,
        case_truth_label=case_truth_label,
        case_log_df=case_log_df,
        template_embedding=template_embedding,
        n_splits=5,
        random_state=42,
        verbose_fold=False,
        min_samples_per_class=MIN_SAMPLES_PER_CLASS,
        rare_class_strategy=RARE_CLASS_STRATEGY,
    )
else:
    raise ValueError(f"Unsupported EVAL_MODE: {EVAL_MODE}")



------------------------------ Eval Mode: supervised ------------------------------
fold 1/2: acc=0.6633, macro_f1=0.4726, weighted_f1=0.6608, macro_recall=0.4415, weighted_recall=0.6633, test_size=3561
fold 1 confusion_matrix labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
[[ 52   5   5   0   4   0   1   3   2   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   3   0   0   0]
 [  0 376  19   3  44   0  12  16   1   0   2   1   0   0   0   0   1   0
    3   0   0   0   0   0   0   4   0   0   0]
 [  0  34 266   0  35   1   5   4   1   0   2  13   0   0   0   0   1   1
    0   2   0   0   0   0   2   4   0   0   0]
 [  0  15   9  76   7   1   2   2   4   0   3   0   0   0   0   0   1   0
    0   0   0   0   0   0   0   0   0   0   0]
 [  0  72  25   0 273   5  16   3  14   5   0   0   0   0   0   0   0   0
    0   0   2   0   0   0   0   4   0   1   0]
 [  0  14   1   1  12  44   3   1   0   1   4   0   0

In [11]:
# Intermediate trace: how raw logs become embeddings
from IPython.display import display


def _prepare_eval_subset(case_name_list, case_truth_label, case_log_df,
                         min_samples_per_class=2, rare_class_strategy='drop'):
    class_counts = Counter(case_truth_label.tolist())
    rare_labels = [label for label, cnt in class_counts.items() if cnt < min_samples_per_class]

    if rare_labels:
        if rare_class_strategy != 'drop':
            raise ValueError(
                f'Not enough samples per class for StratifiedKFold: '
                f'rare_labels={rare_labels}, min_samples_per_class={min_samples_per_class}'
            )
        keep_indices = [idx for idx, y in enumerate(case_truth_label.tolist()) if y not in rare_labels]
        if len(keep_indices) == 0:
            raise ValueError('All samples are dropped by rare class filtering.')
        case_name_list = [case_name_list[idx] for idx in keep_indices]
        case_truth_label = case_truth_label[keep_indices]
        case_log_df = {name: case_log_df[name] for name in case_name_list}
        class_counts = Counter(case_truth_label.tolist())

    min_class_count = min(class_counts.values())
    effective_splits = min(5, min_class_count)
    if effective_splits < 2:
        raise ValueError(f'Not enough samples per class after filtering: min_class_count={min_class_count}')

    return case_name_list, case_truth_label, case_log_df, class_counts, effective_splits


def _build_case_template_table(case_log_df_one, model, split_name):
    template_sequence = case_log_df_one['EventId'].astype(str).tolist()
    template_counter = Counter(template_sequence)
    train_template_set = set(model.template_list)

    keep_meta = {}
    unseen_in_train_count = 0
    for template, count in template_counter.items():
        if split_name == 'test' and template not in train_template_set:
            keep_meta[template] = (False, np.nan, 'unseen_in_train')
            unseen_in_train_count += count
            continue

        idf = float(model.template_idf.get(template, 0.0))
        if idf > 0:
            keep_meta[template] = (True, idf, 'kept')
        else:
            keep_meta[template] = (False, idf, 'idf_filtered')

    important_log_count = int(sum(
        template_counter[t] for t, (keep, _, _) in keep_meta.items() if keep
    ))

    template_text_map = {}
    if 'EventTemplate' in case_log_df_one.columns:
        for eid, tpl in zip(
            case_log_df_one['EventId'].astype(str).tolist(),
            case_log_df_one['EventTemplate'].astype(str).tolist(),
        ):
            if eid not in template_text_map:
                template_text_map[eid] = tpl[:100]

    rows = []
    for template, count in template_counter.items():
        keep, idf, status = keep_meta[template]
        tf = (count / important_log_count) if keep and important_log_count > 0 else 0.0
        tf_idf_weight = (tf * idf) if keep else 0.0
        vector_norm = (
            float(np.linalg.norm(model.template_embedding[template]))
            if keep and template in model.template_embedding
            else np.nan
        )
        rows.append({
            'template': template,
            'count': int(count),
            'status': status,
            'idf': idf if np.isfinite(idf) else np.nan,
            'tf': float(tf),
            'tf_idf_weight': float(tf_idf_weight),
            'template_vec_norm': vector_norm,
            'template_text_head': template_text_map.get(template, ''),
        })

    detail_df = pd.DataFrame(rows).sort_values(
        ['tf_idf_weight', 'count'], ascending=[False, False]
    ).reset_index(drop=True)

    return detail_df, important_log_count, unseen_in_train_count, len(template_sequence), template_sequence


def _print_case_trace(case_name, split_name, case_log_df_one, model, embedding_dict,
                      top_templates=10):
    detail_df, important_count, unseen_count, total_logs, seq = _build_case_template_table(
        case_log_df_one, model, split_name
    )
    case_embedding = np.asarray(embedding_dict[case_name], dtype=float)
    emb_norm = float(np.linalg.norm(case_embedding))

    print(f'[{split_name}] case={case_name}')
    print(
        f'  total_logs={total_logs}, unique_templates={len(detail_df)}, '
        f'kept_logs={important_count}, filtered_logs={total_logs - important_count}, '
        f'unseen_in_train_logs={unseen_count}, embedding_norm={emb_norm:.6f}'
    )
    print(f'  raw EventId head(12): {seq[:12]}')
    display(detail_df.head(top_templates))


if SHOW_INTERMEDIATE_TRACE:
    names_f, labels_f, log_df_f, class_counts_f, effective_splits_f = _prepare_eval_subset(
        case_name_list=case_name_list,
        case_truth_label=case_truth_label,
        case_log_df=case_log_df,
        min_samples_per_class=MIN_SAMPLES_PER_CLASS,
        rare_class_strategy=RARE_CLASS_STRATEGY,
    )

    skf_trace = StratifiedKFold(
        n_splits=effective_splits_f,
        shuffle=True,
        random_state=42,
    )
    fold_splits = list(skf_trace.split(np.arange(len(names_f)), labels_f))

    if len(fold_splits) == 0:
        raise ValueError('No fold generated for intermediate trace.')

    fold_id = int(max(1, min(TRACE_FOLD_INDEX, len(fold_splits))))
    train_index_arr, test_index_arr = fold_splits[fold_id - 1]
    train_index = train_index_arr.tolist()
    test_index = test_index_arr.tolist()

    train_df = {names_f[idx]: log_df_f[names_f[idx]] for idx in train_index}
    test_df = {names_f[idx]: log_df_f[names_f[idx]] for idx in test_index}

    try:
        trace_model = LogKG(train_df, test_df, IDF_threshold, template_embedding, embedding_size=EMBEDDING_SIZE)
    except TypeError:
        trace_model = LogKG(train_df, test_df, IDF_threshold, template_embedding)

    trace_model.get_train_embedding()
    trace_model.get_test_embedding()

    labels_order = np.unique(labels_f)

    print('=' * 28, 'Intermediate Trace', '=' * 28)
    print(f'eval_mode={EVAL_MODE}, fold={fold_id}/{effective_splits_f}')
    print(f'train_size={len(train_index)}, test_size={len(test_index)}')
    print('train class dist:', Counter(labels_f[train_index].tolist()))
    print('test class dist :', Counter(labels_f[test_index].tolist()))

    idf_values = np.array(list(trace_model.template_idf.values()), dtype=float)
    kept_templates = int(np.sum(idf_values > 0))
    filtered_templates = int(np.sum(idf_values == 0))
    print(
        f'template stats: total={len(idf_values)}, kept_by_idf={kept_templates}, '
        f'filtered_by_idf={filtered_templates}, idf_threshold={IDF_threshold}'
    )

    idf_df = pd.DataFrame({
        'template': list(trace_model.template_idf.keys()),
        'idf': list(trace_model.template_idf.values()),
    })
    print(f'top {TRACE_TOP_TEMPLATES} templates by IDF:')
    display(idf_df[idf_df['idf'] > 0].sort_values('idf', ascending=False).head(TRACE_TOP_TEMPLATES))

    train_case_names = [names_f[idx] for idx in train_index]
    test_case_names = [names_f[idx] for idx in test_index]

    train_norms = np.array([np.linalg.norm(trace_model.train_embedding_dict[name]) for name in train_case_names])
    test_norms = np.array([np.linalg.norm(trace_model.test_embedding_dict[name]) for name in test_case_names])
    print(f'train zero embeddings: {int(np.sum(train_norms == 0.0))}/{len(train_norms)}')
    print(f'test zero embeddings : {int(np.sum(test_norms == 0.0))}/{len(test_norms)}')

    train_template_set = set(trace_model.template_list)
    total_test_logs = 0
    unseen_test_logs = 0
    for case_name in test_case_names:
        seq = log_df_f[case_name]['EventId'].astype(str).tolist()
        total_test_logs += len(seq)
        unseen_test_logs += sum(1 for t in seq if t not in train_template_set)
    unseen_ratio = (unseen_test_logs / total_test_logs) if total_test_logs > 0 else 0.0
    print(f'test unseen-template logs: {unseen_test_logs}/{total_test_logs} ({unseen_ratio:.2%})')

    print('-' * 20, f'Train case traces (top {TRACE_CASES_PER_SPLIT})', '-' * 20)
    for case_name in train_case_names[:TRACE_CASES_PER_SPLIT]:
        _print_case_trace(
            case_name=case_name,
            split_name='train',
            case_log_df_one=log_df_f[case_name],
            model=trace_model,
            embedding_dict=trace_model.train_embedding_dict,
            top_templates=TRACE_TOP_TEMPLATES,
        )

    print('-' * 20, f'Test case traces (top {TRACE_CASES_PER_SPLIT})', '-' * 20)
    for case_name in test_case_names[:TRACE_CASES_PER_SPLIT]:
        _print_case_trace(
            case_name=case_name,
            split_name='test',
            case_log_df_one=log_df_f[case_name],
            model=trace_model,
            embedding_dict=trace_model.test_embedding_dict,
            top_templates=TRACE_TOP_TEMPLATES,
        )

    if EVAL_MODE == 'logkg':
        trace_train_set = np.array([
            trace_model.train_embedding_dict[names_f[idx]] for idx in train_index
        ])
        _, trace_cluster_result = get_logkg_result(
            train_set=trace_train_set,
            train_index=train_index,
            verbose=False,
        )
        print('logkg cluster distribution on traced fold:', Counter(trace_cluster_result))


In [12]:
# Output display: summary metrics, per-fold metrics, confusion matrices
from IPython.display import display

if 'eval_result' not in globals():
    raise ValueError("'eval_result' not found. Please run the evaluation cell first.")

summary_cols = [
    'acc_mean', 'acc_std',
    'macro_f1_mean', 'macro_f1_std',
    'weighted_f1_mean', 'weighted_f1_std',
    'macro_recall_mean', 'macro_recall_std',
    'weighted_recall_mean', 'weighted_recall_std',
    'effective_splits',
]
summary_row = {k: eval_result.get(k, None) for k in summary_cols if k in eval_result}
print('summary:')
display(pd.DataFrame([summary_row]))

fold_count = len(eval_result.get('acc_list', []))
if fold_count > 0:
    fold_df = pd.DataFrame({
        'fold': np.arange(1, fold_count + 1),
        'acc': eval_result.get('acc_list', [None] * fold_count),
        'macro_f1': eval_result.get('macro_f1_list', eval_result.get('f1_list', [None] * fold_count)),
        'weighted_f1': eval_result.get('weighted_f1_list', [None] * fold_count),
        'macro_recall': eval_result.get('macro_recall_list', [None] * fold_count),
        'weighted_recall': eval_result.get('weighted_recall_list', [None] * fold_count),
    })
    print('per-fold metrics:')
    display(fold_df)

if 'final_class_dist' in eval_result:
    print('final_class_dist:', eval_result['final_class_dist'])

labels = eval_result.get('labels', [])
if hasattr(labels, 'tolist'):
    labels = labels.tolist()

fold_cms = eval_result.get('fold_confusion_matrices', [])
if fold_cms and labels:
    for fold_idx, fold_cm in enumerate(fold_cms, start=1):
        fold_cm_df = pd.DataFrame(
            fold_cm,
            index=[f'true_{label}' for label in labels],
            columns=[f'pred_{label}' for label in labels],
        )
        print(f'fold {fold_idx} confusion matrix:')
        display(fold_cm_df)

if labels and 'confusion_matrix' in eval_result:
    cm_df = pd.DataFrame(
        eval_result['confusion_matrix'],
        index=[f'true_{label}' for label in labels],
        columns=[f'pred_{label}' for label in labels],
    )
    print('overall confusion matrix (all folds merged):')
    display(cm_df)

def _json_safe(obj):
    if hasattr(obj, 'tolist'):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    raise TypeError(f'Object of type {type(obj).__name__} is not JSON serializable')

result_dir = os.path.join('result')
os.makedirs(result_dir, exist_ok=True)
result_path = os.path.join(
    result_dir,
    f'os_eval_{os.path.basename(OS_PREPROCESSED_DIR)}_{LABEL_LEVEL}_{EVAL_MODE}.json'
)
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(eval_result, f, ensure_ascii=False, indent=2, default=_json_safe)
print('saved eval_result:', result_path)


summary:


,acc_mean,acc_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,macro_recall_mean,macro_recall_std,weighted_recall_mean,weighted_recall_std,effective_splits
0,0.666339,0.003043,0.484419,0.011771,0.664344,0.003562,0.446843,0.005376,0.666339,0.003043,2


per-fold metrics:


,fold,acc,macro_f1,weighted_f1,macro_recall,weighted_recall
0,1,0.663297,0.472647,0.660782,0.441467,0.663297
1,2,0.669382,0.496190,0.667906,0.452219,0.669382


final_class_dist: {0: 149, 1: 965, 2: 741, 3: 240, 4: 841, 5: 178, 6: 767, 7: 1459, 8: 396, 9: 200, 10: 92, 11: 28, 12: 89, 13: 61, 14: 7, 15: 10, 16: 137, 17: 95, 18: 98, 19: 82, 20: 91, 21: 15, 22: 7, 23: 3, 24: 62, 25: 295, 26: 2, 27: 9, 28: 2}
fold 1 confusion matrix:


...

fold 2 confusion matrix:


,pred_0,pred_1,pred_2,pred_3,pred_4,pred_5,pred_6,pred_7,pred_8,pred_9,...,pred_19,pred_20,pred_21,pred_22,pred_23,pred_24,pred_25,pred_26,pred_27,pred_28
true_0,44,7,9,1,10,0,2,0,0,0,...,0,0,0,0,0,0,1,0,0,0
true_1,0,379,17,2,20,8,27,5,3,2,...,3,1,0,0,0,1,1,0,0,0
true_2,0,37,260,0,27,1,8,11,3,1,...,1,0,0,0,0,0,5,0,4,0
true_3,0,22,8,73,4,8,1,2,1,0,...,0,1,0,0,0,0,0,0,0,0
true_4,0,70,26,0,262,1,6,9,29,2,...,0,4,0,0,0,0,1,0,0,0
true_5,0,18,2,0,13,45,0,2,2,2,...,0,3,0,0,0,0,1,0,0,0
true_6,1,40,8,0,33,1,267,12,12,3,...,0,1,0,0,0,0,0,0,0,0
true_7,0,27,39,0,19,0,4,602,22,0,...,0,0,0,0,0,0,3,0,0,0
true_8,0,6,5,0,18,0,8,18,141,0,...,0,0,0,0,0,0,0,0,0,0
true_9,0,13,3,0,23,3,10,2,0,43,...,0,0,0,0,0,0,0,0,0,0


overall confusion matrix (all folds merged):


,pred_0,pred_1,pred_2,pred_3,pred_4,pred_5,pred_6,pred_7,pred_8,pred_9,...,pred_19,pred_20,pred_21,pred_22,pred_23,pred_24,pred_25,pred_26,pred_27,pred_28
true_0,96,12,14,1,14,0,3,3,2,0,...,0,0,0,0,0,0,4,0,0,0
true_1,0,755,36,5,64,8,39,21,4,2,...,3,1,0,0,0,1,5,0,0,0
true_2,0,71,526,0,62,2,13,15,4,1,...,3,0,0,0,0,2,9,0,4,0
true_3,0,37,17,149,11,9,3,4,5,0,...,0,1,0,0,0,0,0,0,0,0
true_4,0,142,51,0,535,6,22,12,43,7,...,0,6,0,0,0,0,5,0,1,0
true_5,0,32,3,1,25,89,3,3,2,3,...,0,7,0,0,0,0,1,0,0,0
true_6,1,76,23,0,69,1,517,22,28,13,...,1,1,0,0,0,0,0,0,0,0
true_7,0,57,73,0,56,0,4,1174,58,0,...,0,0,0,0,0,0,14,0,0,0
true_8,0,9,13,0,49,0,14,18,287,0,...,0,0,0,2,0,0,0,0,0,0
true_9,0,23,8,0,45,3,14,2,2,99,...,0,1,0,0,0,0,0,0,0,0


saved eval_result: result\os_eval_OS_preprocessed_B507_minor_supervised.json
